# Introdu??o

**Resumo:**

O exemplo aqui desenvolvido tem como objetivo apresentar conceitos iniciais de implementa??o de redes neurais com python e tensorflow/keras. Esse modelo apresenta um modelo MLP b?sico que pode ser expandido mudando o n?mero de neur?nios e camadas. Em adapta??es mais avan?adas, pode-se estudar possibilidade de otimiza??o de hyperpar?metros e outras t?cnincas como aumento de dados.

**N?o ? objetivo nosso desenvolver e otimizar o modelo de classifica??o**. O exemplo tem objetivo meramente did?tico.

---

**Para saber mais:**
* [Palestras e cursos do Ocean](http://www.oceanbrasil.com/)
* Fran?ois Chollet. Deep Learning with Python. Manning Publications, 2017.
* Ian Goodfellow and Yoshua Bengio and Aaron Courville. [Deep Learning](https://www.deeplearningbook.org/). MIT Press, 2016.

# Bibliotecas e Dados

In [ ]:
# Bibliotecas necess?rias
# Manipula??o de dados
import pandas as pd
# Redes Neurais
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import RMSprop
# Plot
import matplotlib.pyplot as plt
#%matplotlib inline
# Avalia??o
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Lendo o dataset Kaggle
train = pd.read_csv("../input/digit-recognizer/train.csv")
# Analisando o dataset
print("Quantidade de elementos de treino: {}". format(len(train)))
print(train.head())

In [ ]:
# Separando x_train e y_train
Y = train["label"]
X = train.drop(labels = ["label"],axis = 1)# S? as colunas pixel<nro>

In [ ]:
print(Y[100])

In [ ]:
# Numtendi nada!
# Bora ver com matplotlib
plt.imshow(X.values[100].reshape(28,28), cmap=plt.cm.binary)
plt.show()

In [ ]:
# Transformando a imagem 2d em um numpy array (imagem 28*28 = 784 pixels)
x = X.values.reshape(42000, 784)

#Normalizando para valores entre 0 e 1
x = x.astype('float32')
x /= 255

print(x[0])

In [ ]:
# Vamos ajustar o formato da saida
num_classes = 10

# Convertendo para um vetor de saida com 10 dimensoes
# ex. 8 => [0,0,0,0,0,0,0,0,1,0]
y = keras.utils.to_categorical(Y, num_classes)
print(y[100])

In [ ]:
# Separando uma parte para treino (90%) e outra para valida??o (10%)
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size = 0.1, random_state=9)
print('Qtde de treino: {}'.format(len(x_train)))
print('Qtde de valida??o: {}'.format(len(x_val)))

# Criando e treinando o Modelo

In [ ]:
# Criando o modelo Sequential
# Sequential: Modelo Keras de ir adicionando camadas (como um lego)
# Dense: Camada onde todas as entradas est?o conectadas em cada neur?nio (totalmente conectada)
# Dropout: Controle de Overfit. Desconecta percentual de neur?nios aleatoriamente.

model = Sequential()
# Camada com 30 neur?nios
model.add(Dense(30, activation='relu', input_shape=(784,)))
# Controle de Overfit
model.add(Dropout(0.2))
# Camada de 20 neur?nios
model.add(Dense(20, activation='relu'))
# Camada de classifica??o final, com 1 neur?nio para cada classe de sa?da. Softmax divide a probabilidade de cada classe.
model.add(Dense(num_classes, activation='softmax'))

model.summary()

In [ ]:
# Compila o modelo
model.compile(loss='categorical_crossentropy',
              optimizer=RMSprop(),
              metrics=['accuracy'])

In [ ]:
# Treina com os parte dos dados
batch_size = 32
epochs = 10
history = model.fit(x_train, y_train,
                    batch_size=batch_size,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_val, y_val))

In [ ]:
#Vamos ver como foi o treino?

fig, ax = plt.subplots(1,2, figsize=(16,8))
ax[0].plot(history.history['loss'], color='b', label="Training loss")
ax[0].plot(history.history['val_loss'], color='r', label="validation loss",axes =ax[0])
legend = ax[0].legend(loc='best', shadow=True)

ax[1].plot(history.history['accuracy'], color='b', label="Training accuracy")
ax[1].plot(history.history['val_accuracy'], color='r',label="Validation accuracy")
legend = ax[1].legend(loc='best', shadow=True)

In [ ]:
# Testa
score = model.evaluate(x_val, y_val, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
# Testando uma entrada qualquer
print(y_train[10])
print(model.predict(x_train[10].reshape((1,784))))
print(model.predict_classes(x_train[10].reshape((1,784))))

# Avaliando o Modelo

In [ ]:
# Vendo alguns reports
# Usando sklearn
import numpy as np

# Classificando toda base de teste
y_pred = model.predict_classes(x_val)
# voltando pro formato de classes
y_test_c = np.argmax(y_val, axis=1)
target_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

In [ ]:
import seaborn as sns

#Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred)

# Plotar a matriz de confus?o com seaborn
plt.figure(figsize=(10, 7))  # Definir o tamanho da figura
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)

# Configura??es adicionais para os eixos
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')

plt.show()


In [ ]:
print('Classification Report')
print(classification_report(y_test_c, y_pred, target_names=target_names))

# Gerando Sa?da (Competi??o Kaggle)

Esse dataset ? usado em uma competi??o tutorial do Kaggle. Caso voc? queira submeter resultados dos seus modelos, basta gerar as predi??es dos dados de testes em um arquivo e fazer upload em: 

In [ ]:
# Gerando sa?da para dataset de teste

#Carrega dataset de teste
test = pd.read_csv("../input/digit-recognizer/test.csv")
print("Qtde de testes: {}".format(len(test)))
# Bota no formato numpy e normaliza
x_test = test.values.reshape(len(test),784)
x_test = x_test.astype('float32')
x_test /= 255

# Faz classifica??o para dataset de teste
y_pred = model.predict_classes(x_test)

# Verficando algum exemplo
i = 0
plt.imshow(test.values[i].reshape(28,28), cmap=plt.cm.binary)
plt.show()
print('Previsto: {}'.format(y_pred[i]))

# Botando no formato de sa?da (competi??o Kaggle)
results = pd.Series(y_pred,name="Label")
submission = pd.concat([pd.Series(range(1,len(y_pred)+1),name = "ImageId"),results],axis = 1)
print(submission.head(10))
#Salvando Arquivo
submission.to_csv("mlp_mnist_v1.csv",index=False)